# Mini-Ablation 3: No Scale - Smoke Test (CPU)

**Ablation ID**: `3_no_scale`

**Description**: Test if scale redundant with mosaic (expect 78-82% Dice)

**Hypothesis**: Mosaic provides sufficient scale diversity via 4-image combination

**Smoke Test Configuration**:
- Dataset: TINY (2 test images, 4 train images)
- Training: 1 epoch, batch=4 (CPU-compatible)
- Expected Runtime: ~3 minutes
- Purpose: check the augmentation configuration before GPU deployment

**Reference**: `docs/BASELINE_VS_CLINICAL_AUGMENTATION_COMPARISON.md` holds the mini-ablation specifications. Not in the repo.


In [ ]:
# NumPy Version Locking
import numpy as np
import subprocess

default_numpy = np.__version__
print(f"Default NumPy: {default_numpy}")

# Install ultralytics (may try to upgrade NumPy)
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

# Force reinstall to prevent 2.x upgrade
subprocess.run(
    ["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"],
    check=True
)

print(f"Locked NumPy: {np.__version__}")


In [ ]:
import sys
from pathlib import Path
from ultralytics import YOLO
import cv2

print(f"Python: {sys.version}")
print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")


In [ ]:
# Environment Setup
IN_KAGGLE = Path('/kaggle/input').exists()

if IN_KAGGLE:
    BASE_INPUT_PATH = Path('/kaggle/input')
    OUTPUT_PATH = Path('/kaggle/working')
    CODE_PATH = BASE_INPUT_PATH / 'wound-seg-code-v1'
    BASELINE_PATH = BASE_INPUT_PATH / 'baseline-outputs'
    TINY_PATH = BASE_INPUT_PATH / 'tiny-wound-seg-smoke-test'
else:
    print("ERROR: Must run on Kaggle with required datasets attached")
    sys.exit(1)

sys.path.append(str(CODE_PATH))

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Code path: {CODE_PATH}")
print(f"Baseline path: {BASELINE_PATH}")
print(f"TINY path: {TINY_PATH}")


In [ ]:
# Smoke Test Parameters
ablation_id = "3_no_scale"

SMOKE_TEST_PARAMS = {
    'epochs': 1,
    'batch': 4,
    'imgsz': 512,
    'patience': 10,
    'seed': 42,
}

print(f'Ablation ID: {ablation_id}')
print('\nSmoke Test Configuration:')
for key, value in SMOKE_TEST_PARAMS.items():
    print(f'  {key}: {value}')


In [ ]:
# Augmentation Configuration
# Mini-Ablation 3: No Scale
# Hypothesis: Mosaic provides sufficient scale diversity via 4-image combination

AUGMENTATION_PARAMS = {
    "auto_augment": "randaugment",
    "bgr": 0.0,
    "close_mosaic": 10,
    "copy_paste": 0.0,
    "degrees": 0.0,
    "erasing": 0.4,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "mixup": 0.0,
    "mosaic": 1.0,
    "perspective": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "translate": 0.1,
}

print('Augmentation Configuration:')
for key, value in sorted(AUGMENTATION_PARAMS.items()):
    print(f'  {key}: {value}')


In [ ]:
# YOLO Configuration
config_content = f'''
path: {TINY_PATH}
train: train_images
val: test_images

names:
  0: background
  1: wound
'''

config_path = OUTPUT_PATH / 'wound_dataset.yaml'
config_path.write_text(config_content.strip())

print(f"YOLO config created: {config_path}")
print("\nConfig content:")
print(config_content)


In [ ]:
# Model Initialization
model = YOLO('yolov8n-seg.pt')
print(f"Model: {model.model_name}")
print(f"Task: {model.task}")


In [ ]:
# Training with Augmentation Configuration
import time

start_time = time.time()

results = model.train(
    data=str(config_path),
    # Smoke test parameters (CPU-compatible)
    batch=SMOKE_TEST_PARAMS['batch'],
    epochs=SMOKE_TEST_PARAMS['epochs'],
    imgsz=SMOKE_TEST_PARAMS['imgsz'],
    patience=SMOKE_TEST_PARAMS['patience'],
    seed=SMOKE_TEST_PARAMS['seed'],
    # Augmentation parameters (mini-ablation specific)
    **AUGMENTATION_PARAMS,
    # Output configuration
    save=True,
    project=str(OUTPUT_PATH / "yolo_training"),
    name="smoke_test",
    exist_ok=True,
    verbose=True,
)

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed/60:.1f} minutes")


In [ ]:
# Smoke Test Validation
import json

validation = {
    "ablation_id": ablation_id,
    "status": "passed" if results else "failed",
    "augmentation_params": AUGMENTATION_PARAMS,
    "smoke_test_params": SMOKE_TEST_PARAMS,
    "training_epochs": results.epoch if results else 0,
}

output_file = OUTPUT_PATH / f"{ablation_id}_smoke_test_validation.json"
output_file.write_text(json.dumps(validation, indent=2))

print("\nSmoke Test Validation:")
print(json.dumps(validation, indent=2))


In [ ]:
# Completion Status
completion = {
    "status": "complete",
    "ablation_id": ablation_id,
    "smoke_test": True,
}

completion_file = OUTPUT_PATH / "completion_status.json"
completion_file.write_text(json.dumps(completion, indent=2))

print("\nSmoke test complete!")
print(f"Validation: {output_file}")
